In [46]:
import numpy as np
import pandas as pd
import polars as pl
import torch 
import torch.nn as nn
from sklearn.impute import SimpleImputer
import kaggle_evaluation.default_inference_server
import os
from sklearn.linear_model import ElasticNet

In [47]:
def new_features(df):
    df = df.copy()
    df['lagged_forward_returns'] = df['forward_returns'].shift(1)
    df['lagged_risk_free_rate'] = df['risk_free_rate'].shift(1)
    df['lagged_market_forward_excess_returns'] = df['market_forward_excess_returns'].shift(1)

    return df

In [48]:

train_set = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
train_set = new_features(train_set)
train_data = train_set[1005:-180]
test_data = train_set[-180:]
print(train_set.shape)
print(train_data.shape)
print(test_data.shape)

(8990, 101)
(7805, 101)
(180, 101)


In [49]:
train_data.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,date_id,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,V6,V7,V8,V9,forward_returns,risk_free_rate,market_forward_excess_returns,lagged_forward_returns,lagged_risk_free_rate,lagged_market_forward_excess_returns
1005,1005,0,0,0,1,0,-1,0,0,0,...,NaN,NaN,NaN,NaN,0.006725,0.000122,0.006293,-0.003350,0.000121,-0.003781
1006,1006,0,0,0,1,0,-1,0,0,0,...,0.000661,NaN,0.000661,NaN,-0.000669,0.000121,-0.001100,0.006725,0.000122,0.006293
1007,1007,0,0,0,1,0,-1,0,0,0,...,0.000661,NaN,0.000661,NaN,0.005348,0.000121,0.004917,-0.000669,0.000121,-0.001100
1008,1008,0,0,0,1,0,-1,0,0,0,...,0.000661,NaN,0.000661,NaN,0.001996,0.000121,0.001565,0.005348,0.000121,0.004917
1009,1009,0,0,0,1,0,-1,0,0,0,...,0.000661,NaN,0.000661,NaN,-0.001327,0.000121,-0.001758,0.001996,0.000121,0.001565


In [50]:
train_data.columns

Index(['date_id', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9',
       ...
       'V6', 'V7', 'V8', 'V9', 'forward_returns', 'risk_free_rate',
       'market_forward_excess_returns', 'lagged_forward_returns',
       'lagged_risk_free_rate', 'lagged_market_forward_excess_returns'],
      dtype='object', length=101)

In [51]:
features = [col for col in train_data.columns
           if col not in['date_id','forward_returns',
       'risk_free_rate', 'market_forward_excess_returns']]
print(features)

['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'M1', 'M10', 'M11', 'M12', 'M13', 'M14', 'M15', 'M16', 'M17', 'M18', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'P1', 'P10', 'P11', 'P12', 'P13', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'S1', 'S10', 'S11', 'S12', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'V1', 'V10', 'V11', 'V12', 'V13', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'lagged_forward_returns', 'lagged_risk_free_rate', 'lagged_market_forward_excess_returns']


In [65]:
X_train = train_data[features].values
y_train = train_data['forward_returns'].values

X_test = test_data[features].values
y_test = test_data['forward_returns'].values

(7805, 97)

In [53]:
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)

X_test = imputer.transform(X_test)

In [54]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
y_train_scaled = scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_scaled = scaler.transform(y_test.reshape(-1, 1)).ravel()
y_train_scaled

array([ 0.57661801, -0.10565027,  0.44952582, ..., -2.79398005,
       -0.07224278,  1.06437347])

In [55]:

MIN_INVESTMENT = 0.0
MAX_INVESTMENT = 2.0

In [56]:
def convert_to_allocations(predictions):
    # allocations = predictions * 15 + 1.0
    scaled = np.where(predictions > 0, 
                     1.5 + predictions * 50,  # Positive: 1.5 to 2.0
                     0.0 + predictions * 10)
    return np.clip(scaled, MIN_INVESTMENT, MAX_INVESTMENT)

In [57]:
def sharpe_ratio(returns, allocations):
    
    strategy_returns = returns * allocations

    if strategy_returns.std() == 0:
        return 0.0

    daily_mean = strategy_returns.mean()
    daily_std = strategy_returns.std()
    sharpe = (daily_mean / daily_std) * np.sqrt(256)

    return sharpe

In [58]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet
import lightgbm as lgb

In [59]:
from lightgbm import LGBMRegressor

model = LGBMRegressor(
    objective='regression',
    boosting_type='gbdt',
    learning_rate=0.05,
    num_leaves=31,
    max_depth=8,
    n_estimators=1000,
    feature_fraction=0.9,
    bagging_fraction=0.8,
    bagging_freq=5,
    random_state=42
)

model.fit(
    X_train, y_train_scaled,
   

)

[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009920 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 22339
[LightGBM] [Info] Number of data points in the train set: 7805, number of used features: 97
[LightGBM] [I

LGBMRegressor(bagging_fraction=0.8, bagging_freq=5, feature_fraction=0.9,
              learning_rate=0.05, max_depth=8, n_estimators=1000,
              objective='regression', random_state=42)

In [60]:
pred_scaled = model.predict(X_test)
predictions = scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()

allocations = convert_to_allocations(predictions)
sharpe = sharpe_ratio(y_test, allocations)
sharpe

[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8


1.1318432189478262

In [61]:
print(predictions.round(4))

[ 0.0018  0.0002 -0.0014 -0.002   0.003   0.003   0.0012  0.0017  0.001
 -0.001   0.0024  0.0029  0.0055  0.0001  0.0016  0.0018  0.0019  0.0024
  0.0032  0.0005  0.0003 -0.0016  0.0019  0.0002  0.0006  0.0021  0.0002
  0.0025  0.0046  0.0021  0.002   0.0019  0.0007  0.0025  0.0011  0.0038
  0.0047  0.0016  0.0011  0.0018  0.0007  0.0032  0.0059  0.0038  0.0131
  0.0034  0.0126 -0.0007  0.0061  0.0071  0.0057  0.01    0.0081  0.008
  0.0146  0.0067  0.0073  0.0073  0.0076  0.0066  0.0065 -0.0002  0.006
  0.0053  0.0043  0.0043 -0.0015  0.008   0.0068  0.0149  0.0328  0.0131
  0.0224 -0.0059  0.0159  0.0133  0.0074  0.0079  0.007   0.0048  0.0082
  0.0065  0.0049  0.0028  0.0042  0.0049  0.0055  0.0075  0.0049  0.0061
  0.0078  0.0041  0.0044  0.0062  0.0047  0.0029  0.0004  0.0003 -0.0007
 -0.0002  0.003   0.0049  0.0029  0.0041  0.0062  0.0013  0.0047  0.0045
  0.0023  0.0027  0.      0.004   0.0045  0.0027  0.003   0.0008  0.0042
  0.0011  0.0016  0.0015  0.0006  0.0029  0.0053  0.00

In [62]:
y_test.round(4)

array([ 0.006 ,  0.0111,  0.0001, -0.0105, -0.0114, -0.0036, -0.0025,
        0.0125,  0.0058, -0.0113,  0.0015, -0.0153,  0.0016,  0.0014,
        0.0182, -0.0019,  0.01  ,  0.0092,  0.0056,  0.0055, -0.0029,
       -0.0141,  0.0086, -0.0045,  0.0054, -0.0053, -0.0067,  0.0067,
        0.0041,  0.0035, -0.0092,  0.0068,  0.0008, -0.0032,  0.0106,
       -0.    ,  0.0029,  0.0024, -0.0042, -0.0171, -0.0046, -0.005 ,
        0.0005, -0.016 ,  0.0156, -0.0175, -0.0118,  0.0107, -0.0178,
        0.0056, -0.0266, -0.0083,  0.0053, -0.0133,  0.0207,  0.0077,
       -0.0108,  0.0109, -0.0029,  0.0003,  0.0179,  0.0024, -0.0119,
       -0.0027, -0.0201,  0.0067,  0.0028,  0.0063, -0.0396, -0.0396,
       -0.0018, -0.0157,  0.0407, -0.0398,  0.0178,  0.0097, -0.0028,
       -0.0222,  0.0014, -0.0238,  0.026 ,  0.0155,  0.021 ,  0.0072,
        0.0004,  0.0063,  0.0004,  0.0071,  0.0148, -0.0057, -0.0084,
        0.0042,  0.007 , -0.0013,  0.033 ,  0.0066,  0.0013,  0.0049,
        0.0063,  0.0

In [63]:
def predict(test: pl.DataFrame):
    test_pd = test.to_pandas()
    # print(test_pd.columns)

    test_features = test_pd[features]
    # print(test_features.columns)
    # print(test_features.values)
    test_features.values
    test_features = imputer.transform(test_features)
    
    raw_pred = model.predict(test_features)[0]
    allocation = convert_to_allocations(np.array([raw_pred]))[0]
    
    return float(allocation)


In [64]:
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))

[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will b

/usr/local/lib/python3.11/dist-packages/sklearn/base.py:432: UserWarning: X has feature names, but SimpleImputer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:432: UserWarning: X has feature names, but SimpleImputer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:432: UserWarning: X has feature names, but SimpleImputer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:432: UserWarning: X has feature names, but SimpleImputer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:432: UserWarning: X has feature names, but SimpleImputer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:432: UserWarning: X has feature names, but SimpleImputer was fitted without feature names
  warnings.warn(
/usr/local/lib/python3